# PageIndex RAG Demo
## Chat with Your Documents Using AI — No Vector Database Needed!

**PageIndex** is a new kind of document AI that works like a human expert. Instead of breaking your PDF into chunks and doing keyword search, it builds a **reasoning tree** (like a smart table of contents) and uses an LLM to *navigate* it — just like how you'd skim a book's index to find an answer.

### What we'll build in this notebook:
1. Upload a PDF to PageIndex
2. Build a reasoning tree from the document
3. Search the tree intelligently using an LLM
4. Get accurate, cited answers to any question

> **Before you start:** Add your `PAGEINDEX_API_KEY` and `OPENAI_API_KEY` to a `.env` file in this folder.

In [1]:
%pip install -q --upgrade pageindex

Note: you may need to restart the kernel to use updated packages.


## Step 1: Install PageIndex

We install the `pageindex` Python library. The `-q` flag keeps the output clean.

> **Note:** If you see a message asking you to restart the kernel, do that before running the next cell.

In [1]:
import os 
from pageindex import PageIndexClient, utils
from dotenv import load_dotenv

load_dotenv(override=True) 

# Get your PageIndex API key from https://dash.pageindex.ai/api-keys
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

## Step 2: Set Up the LLM (OpenAI)

We import OpenAI and define a helper function `call_llm()`. This function takes a prompt, sends it to GPT, and returns the response as a string. We'll reuse this function throughout the notebook — for both searching the tree and generating the final answer.

In [2]:
import openai

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

async def call_llm(prompt, model="gpt-4o", temperature=0):
    client = openai.AsyncOpenAI(
        base_url="https://demogpt4o-6358-resource.openai.azure.com/openai/v1/",
        api_key=OPENAI_API_KEY
    )
    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

## Step 3: Import Libraries & Connect to PageIndex

Here we load our API keys from the `.env` file and create a `PageIndexClient`. Think of this as opening a connection to PageIndex — all document uploads, status checks, and tree retrievals go through `pi_client`.

In [3]:
pdf_path = "HR_Policies.pdf"

doc_id = pi_client.submit_document(pdf_path)["doc_id"]
print('Document Submitted:', doc_id)

Document Submitted: pi-cmnby7dfb051609qskickxr99


## Step 4: Upload Your PDF

We point to our PDF file and submit it to PageIndex using `submit_document()`. PageIndex reads the document and starts building the reasoning tree in the background. We get back a `doc_id` — a unique ID for our document that we use in all future steps.

In [4]:
import time

print("Waiting for document to be processed...")
while not pi_client.is_retrieval_ready(doc_id):
    print("Still processing... retrying in 10 seconds")
    time.sleep(10)

tree = pi_client.get_tree(doc_id, node_summary=True)['result']
print('Simplified Tree Structure of the Document:')
utils.print_tree(tree)

Waiting for document to be processed...
Still processing... retrying in 10 seconds
Still processing... retrying in 10 seconds
Simplified Tree Structure of the Document:
[{'title': 'HR Policies Handbook',
  'node_id': '0000',
  'prefix_summary': '# HR Policies Handbook\n\nDetailed policy ...',
  'nodes': [{'title': 'Use of this handbook',
             'node_id': '0001',
             'summary': '## Use of this handbook\n\nThis document e...'},
            {'title': 'Introduction',
             'node_id': '0002',
             'summary': '## Introduction\n\nThis handbook sets out ...'},
            {'title': 'Company Values &amp; Culture',
             'node_id': '0003',
             'summary': '## Company Values &amp; Culture\n\nA healt...'},
            {'title': 'Recruitment &amp; Hiring',
             'node_id': '0004',
             'summary': '## Recruitment &amp; Hiring\n\nHiring deci...'},
            {'title': 'Employee Onboarding',
             'node_id': '0005',
             'sum

## Step 5: Wait for Processing & Retrieve the Document Tree

PageIndex needs a moment to analyze the document and build its tree. This cell automatically polls every 10 seconds until it's ready — no manual refreshing needed.

Once done, it prints the **reasoning tree** — PageIndex's hierarchical summary of your document. Each node has an ID, a title, and a summary. This is what the LLM will navigate to answer your questions.

In [5]:
import json

query = "What are the consequences for sexual harrasment?"

tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = await call_llm(search_prompt)
print(tree_search_result)

{
    "thinking": "The question asks about the consequences for sexual harassment. The relevant nodes would be those that discuss harassment policies, disciplinary actions, and grievance handling, as these sections likely outline the consequences and procedures for addressing such behavior.",
    "node_list": ["0019", "0021", "0022"]
}


## Step 6: Search the Document Tree with an LLM

This is where PageIndex shines! Instead of doing a simple keyword search, we send the **entire tree structure** (without full text — just titles and summaries) to the LLM and ask it: *"Which sections are relevant to this question?"*

The LLM reasons through the tree and returns a list of relevant node IDs along with its thinking process. This is fully transparent — you can see exactly *why* each section was selected.

In [6]:
node_map = utils.create_node_mapping(tree)
tree_search_result_json = json.loads(tree_search_result)

print('Reasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Reasoning Process:
The question asks about the consequences for sexual harassment. The relevant nodes would be those
that discuss harassment policies, disciplinary actions, and grievance handling, as these sections
likely outline the consequences and procedures for addressing such behavior.

Retrieved Nodes:
Node ID: 0019	 Page: 7	 Title: Anti-Harassment Policy
Node ID: 0021	 Page: 7	 Title: Grievance Handling
Node ID: 0022	 Page: 8	 Title: Disciplinary Actions


## Step 7: Review the Retrieved Nodes

Here we display the LLM's **reasoning process** and the **list of document sections** it selected. For each node you can see:
- The **Node ID** (used to fetch the actual text)
- The **page number** in the original PDF
- The **section title**

This explainability is one of PageIndex's key advantages — you always know where the answer is coming from.

In [7]:
node_list = tree_search_result_json["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await call_llm(answer_prompt)
utils.print_wrapped(answer)

Retrieved Context:

## Anti-Harassment Policy

Harassment of any kind is prohibited. This includes unwelcome conduct based on protected
characteristics, sexual harassment, bullying, intimidation, threats, or repeated behavior that
creates an offensive, hostile, or humiliating environment.

A single serious incident may be enough to trigger action. The organization encourages early
reporting so that concerns can be addressed before they escalate and so that retaliation can be
prevented.

Investigations will be handled promptly, confidentially to the extent possible, and impartially.
Confirmed violations may lead to corrective action up to and including termination.

- Do not make offensive jokes, comments, messages, or gestures.
- Report concerns without delay, even if they involve a senior person.
- Retaliation against complainants or witnesses is prohibited.


## Grievance Handling

Employees are encouraged to raise work-related concerns early and through the proper channel. A
grievan

## Step 8: Generate the Final Answer

Now we fetch the **actual text** from only the relevant nodes and pass it to the LLM as context. The LLM then generates a clear, concise answer — grounded entirely in the document content. No guessing, no hallucinations.

In [12]:
async def ask(query):
    tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

    search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

    search_result = await call_llm(search_prompt)
    search_result_json = json.loads(search_result)

    node_list = search_result_json["node_list"]
    relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

    answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

    answer = await call_llm(answer_prompt)

    print(f"Query: {query}")
    print(f"\nRelevant Nodes: {node_list}")
    print("\nAnswer:")
    utils.print_wrapped(answer)
    print("-" * 60)

# --- Enter your query below and run this cell ---
user_query = input("Enter your query: ")
await ask(user_query)

Query: Provide details on Induction

Relevant Nodes: ['0005', '0012']

Answer:
Induction is part of the onboarding process and involves providing policy orientation and role-
specific information to new employees. It helps them understand the organization, their role, key
systems, and expected standards of behavior and performance. HR and the hiring manager should
prepare induction materials before the start date, and a manager or buddy should be assigned for
practical support. Induction may also include compliance learning, technical upskilling, and role-
specific training.
------------------------------------------------------------


## Step 9: Ask Anything!

The `ask()` function bundles the entire pipeline into one call — tree search, context retrieval, and answer generation. Just run the cell, type your question, and press Enter.

**Try asking:**
- *"What happens if an employee violates the email policy?"*
- *"How do I file a grievance?"*
- *"What are the penalties for sexual harassment?"*
- *"Can I use company internet for personal use?"*

In [ ]:
await ask(user_query)